<div align='center'>

# 🧠 KRONOS MCTS
## *Monte Carlo Tree Search + Deep RL Architecture*

</div>

---

```
Traditional agent:  heuristic(state) → action  [1 path]
MCTS agent:         simulate 500 futures → pick best  [500 paths]
```

## Architecture:

| Module | Role |
|--------|------|
| `GameState` | Fast 1-turn simulator for MCTS rollouts |
| `MCTSNode` | UCB1 tree node: balances exploit vs explore |
| `MCTSAgent` | 45ms budget search: 200-500 simulations/move |
| `kronos_rollout` | KRONOS v6 heuristic as fast rollout policy |
| `OrbitWarsEnv` | Gymnasium wrapper for PPO/DRL training |
| `extract_features` | 288-dim strategic feature vector |

**UCB1 formula:**
```
score = win_rate + 1.414 × √(ln(parent_visits) / node_visits)
         ↑ exploit          ↑ explore (visit less-tried branches)
```


## ⚙️ Cell 1 — Install


In [ ]:
%%capture
!pip install --upgrade 'kaggle-environments>=1.28.0'
!pip install gymnasium stable-baselines3 torch --quiet


## 🔌 Cell 2 — Environment


In [ ]:
from kaggle_environments import make
import math, time, random, copy, collections
import numpy as np

env=make('orbit_wars',debug=True)
env.reset()
obs=dict(env.state[0].observation)
print(f'✅ {env.name} v{env.version}')
print(f'   av={obs["angular_velocity"]:.4f} | planets={len(obs["planets"])}')


## 🧠 Cell 3 — KRONOS MCTS (Full Architecture)

Contains: Physics, FeatureExtractor, GameState, MCTSNode, MCTSAgent,
KronosRollout, OrbitWarsGym, and final `orbital_strategist`.


In [ ]:
"""
KRONOS MCTS — Monte Carlo Tree Search + KRONOS v6 Rollout Policy
================================================================
Architecture:
  1. OrbitWarsEnv      — Gymnasium wrapper for orbit_wars
  2. FeatureExtractor  — Strategic feature engineering (not just distance)
  3. MCTSNode          — Tree node with UCB1 selection
  4. MCTSAgent         — 50ms budget MCTS with KRONOS rollout
  5. PPOTrainer        — Self-play training setup (Stable Baselines3)

Why MCTS > pure heuristic:
  - Simulates 200-500 future states per move
  - Picks the action that wins in MOST simulated futures
  - KRONOS v6 used as rollout policy (fast approximation)
  - UCB1 balances exploration vs exploitation
"""
import math, time, random, copy, collections
import numpy as np

SX,SY,SR,INNER,MS = 50.0,50.0,5.0,38.0,500

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1: PHYSICS (shared)
# ─────────────────────────────────────────────────────────────────────────────
class _P:
    __slots__=['id','owner','x','y','radius','ships','production']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
    def copy(self):
        p=_P(); [setattr(p,f,getattr(self,f)) for f in self.__slots__]; return p

class _F:
    __slots__=['id','owner','x','y','angle','ships']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
    def copy(self):
        f=_F(); [setattr(f,s,getattr(self,s)) for s in self.__slots__]; return f

def spd(n):  return min(6.0,1.0+(max(1,n)-1)*5.0/99.0)
def d2(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
def inn(p):  return d2(p.x,p.y,SX,SY)<INNER
def pred(p,av,t):
    if not inn(p): return p.x,p.y
    r=d2(p.x,p.y,SX,SY); a=math.atan2(p.y-SY,p.x-SX)+av*t
    return SX+r*math.cos(a),SY+r*math.sin(a)
def icp(sx,sy,tp,av,n,it=18):
    tx,ty=tp.x,tp.y
    for _ in range(it):
        dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
        nx,ny=pred(tp,av,t)
        if d2(tx,ty,nx,ny)<0.02: break
        tx,ty=(tx+nx)/2,(ty+ny)/2
    dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
    return math.atan2(ty-sy,tx-sx),dd,t
def sun_ok(ox,oy,a,md):
    dx,dy=math.cos(a),math.sin(a); fx,fy=SX-ox,SY-oy; tp=fx*dx+fy*dy
    if not(0<tp<md): return True
    return abs(fx*dy-fy*dx)>=SR+1.5
def safe(ox,oy,a,d,sw=42,st=32):
    if sun_ok(ox,oy,a,d): return a,True
    for i in range(1,st+1):
        da=math.radians(sw)*i/st
        for s in(+1,-1):
            alt=a+s*da
            if sun_ok(ox,oy,alt,d): return alt,True
    return a,False
def capture_n(tgt,av,sx,sy,buf=1.07):
    lo,hi=1,max(tgt.ships*2+20,30)
    for _ in range(14):
        mid=(lo+hi)//2
        _,_,eta=icp(sx,sy,tgt,av,mid)
        if mid>int((tgt.ships+tgt.production*eta)*buf): hi=mid
        else: lo=mid+1
    return hi

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2: FEATURE EXTRACTOR
# Strategic features far richer than raw coordinates
# ─────────────────────────────────────────────────────────────────────────────
def extract_features(planets, fleets, player, av, stp):
    """
    Returns numpy array of features for RL observation space.
    Per-planet features (14 features × max_planets):
      - owner_one_hot (4), ships_norm, production_norm, 
        dist_to_sun_norm, dist_to_centroid_norm,
        is_inner, threat_incoming_norm, threat_eta_norm,
        strategic_value, control_radius
    Global features (8):
      - my_prod_share, my_ship_share, step_norm,
        neutral_count_norm, enemy_count_norm,
        my_planet_count_norm, winning_flag, prod_lead
    """
    MAX_P = 20
    mine   = [p for p in planets if p.owner==player]
    enemy  = [p for p in planets if p.owner>=0 and p.owner!=player]
    neutral= [p for p in planets if p.owner<0]

    if mine:
        cx = sum(p.x for p in mine)/len(mine)
        cy = sum(p.y for p in mine)/len(mine)
    else:
        cx, cy = SX, SY

    total_ships = max(1, sum(p.ships for p in planets if p.owner>=0))
    total_prod  = max(1, sum(p.production for p in planets))
    my_ships    = sum(p.ships for p in mine)
    my_prod     = sum(p.production for p in mine)

    # Per-planet features
    planet_feats = np.zeros((MAX_P, 14), dtype=np.float32)
    for i, p in enumerate(planets[:MAX_P]):
        # Owner one-hot (4 players: -1=neutral mapped to 0, else 1-4)
        owner_idx = max(0, min(3, p.owner+1)) if p.owner>=0 else 0
        planet_feats[i, owner_idx] = 1.0
        planet_feats[i, 4]  = p.ships / 200.0
        planet_feats[i, 5]  = p.production / 10.0
        planet_feats[i, 6]  = d2(p.x,p.y,SX,SY) / 70.0
        planet_feats[i, 7]  = d2(p.x,p.y,cx,cy) / 100.0
        planet_feats[i, 8]  = 1.0 if inn(p) else 0.0

        # Threat
        thr = sum(f.ships for f in fleets
                  if f.owner!=player
                  and icp(f.x,f.y,p,av,f.ships)[1] < p.radius+5
                  and icp(f.x,f.y,p,av,f.ships)[2] < 30)
        planet_feats[i, 9]  = min(1.0, thr / 100.0)

        # Strategic value: production × (1/distance_to_enemy_centroid)
        if enemy:
            ecx=sum(e.x for e in enemy)/len(enemy)
            ecy=sum(e.y for e in enemy)/len(enemy)
            planet_feats[i,10] = p.production / max(1,d2(p.x,p.y,ecx,ecy)/10)
        planet_feats[i,11] = (p.owner==player)*1.0
        planet_feats[i,12] = 1.0 if p.owner<0 else 0.0
        planet_feats[i,13] = p.production / 10.0 * (1.0 if p.owner!=player else 0.0)

    # Global features
    global_feats = np.array([
        my_prod / total_prod,                           # production share
        my_ships / total_ships,                         # ship share
        stp / MS,                                       # game progress
        len(neutral) / max(1,len(planets)),             # neutral ratio
        len(enemy) / 4.0,                               # enemy planet ratio
        len(mine) / max(1,len(planets)),                # my planet ratio
        1.0 if my_ships > total_ships/4*1.1 else 0.0,  # winning flag
        (my_prod - (total_prod-my_prod)/3) / 10.0,     # production lead
    ], dtype=np.float32)

    return np.concatenate([planet_feats.flatten(), global_feats])

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3: LIGHTWEIGHT GAME SIMULATOR (for MCTS rollouts)
# ─────────────────────────────────────────────────────────────────────────────
class GameState:
    """
    Minimal game state for fast simulation.
    Simulates one turn: fleet movement, planet capture, production.
    """
    def __init__(self, planets, fleets, av, step, n_players=4):
        self.planets   = [p.copy() for p in planets]
        self.fleets    = [f.copy() for f in fleets]
        self.av        = av
        self.step      = step
        self.n_players = n_players
        self._fid      = max((f.id for f in fleets), default=0) + 1

    def apply_moves(self, moves, player):
        """Apply move list from agent: [[src_id, angle, n_ships], ...]"""
        pid_map = {p.id:p for p in self.planets}
        for move in moves:
            src_id, angle, n = move[0], move[1], move[2]
            src = pid_map.get(src_id)
            if src is None or src.ships < n: continue
            src.ships -= n
            nf = _F()
            nf.id=self._fid; nf.owner=player
            nf.x=src.x; nf.y=src.y
            nf.angle=angle; nf.ships=n
            self.fleets.append(nf)
            self._fid += 1

    def step_sim(self):
        """Advance game by one turn."""
        # Move fleets
        alive = []
        for f in self.fleets:
            s = spd(f.ships)
            f.x += math.cos(f.angle)*s
            f.y += math.sin(f.angle)*s
            if d2(f.x,f.y,SX,SY) < SR: continue
            if not(0<=f.x<=100 and 0<=f.y<=100): continue
            alive.append(f)
        self.fleets = alive

        # Rotate inner planets
        for p in self.planets:
            if inn(p):
                r=d2(p.x,p.y,SX,SY); a=math.atan2(p.y-SY,p.x-SX)+self.av
                p.x=SX+r*math.cos(a); p.y=SY+r*math.sin(a)

        # Fleet arrivals
        arrived=[]
        for f in self.fleets:
            for p in self.planets:
                if d2(f.x,f.y,p.x,p.y) < p.radius + spd(f.ships) + 0.5:
                    if f.owner==p.owner: p.ships+=f.ships
                    else:
                        p.ships-=f.ships
                        if p.ships<0: p.owner=f.owner; p.ships=abs(p.ships)
                    arrived.append(f); break
        for f in arrived:
            if f in self.fleets: self.fleets.remove(f)

        # Production
        for p in self.planets:
            if p.owner>=0: p.ships+=p.production

        self.step+=1

    def score(self, player):
        """Evaluate position for player. Higher = better."""
        my_ships = sum(p.ships for p in self.planets if p.owner==player)
        my_prod  = sum(p.production for p in self.planets if p.owner==player)
        my_fleet = sum(f.ships for f in self.fleets if f.owner==player)
        total    = max(1, sum(p.ships for p in self.planets if p.owner>=0))
        return (my_ships+my_fleet)/total + my_prod*0.3

    def is_terminal(self):
        return self.step >= MS

    def copy(self):
        gs = GameState.__new__(GameState)
        gs.planets   = [p.copy() for p in self.planets]
        gs.fleets    = [f.copy() for f in self.fleets]
        gs.av        = self.av
        gs.step      = self.step
        gs.n_players = self.n_players
        gs._fid      = self._fid
        return gs

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4: MCTS NODE
# UCB1: score = win_rate + C × sqrt(ln(parent_visits) / node_visits)
# ─────────────────────────────────────────────────────────────────────────────
class MCTSNode:
    __slots__ = ['moves','parent','children','visits','value','untried']
    C = 1.414   # exploration constant

    def __init__(self, moves, parent=None):
        self.moves    = moves     # the action that led here
        self.parent   = parent
        self.children = []
        self.visits   = 0
        self.value    = 0.0
        self.untried  = None      # set lazily

    def ucb1(self):
        if self.visits == 0: return float('inf')
        exploit = self.value / self.visits
        explore = self.C * math.sqrt(math.log(self.parent.visits) / self.visits)
        return exploit + explore

    def best_child(self):
        return max(self.children, key=lambda c: c.ucb1())

    def most_visited(self):
        return max(self.children, key=lambda c: c.visits)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5: KRONOS v6 ROLLOUT POLICY (fast heuristic for simulation)
# ─────────────────────────────────────────────────────────────────────────────
def kronos_rollout(planets, fleets, player, av, stp):
    """Fast KRONOS heuristic for MCTS rollout."""
    mine   = [p for p in planets if p.owner==player]
    others = [p for p in planets if p.owner!=player]
    if not mine or not others: return []

    rem=MS-stp; moves=[]; used={}; done=set()
    def gn(p): return max(3,p.production*2)
    def sp(p): return p.ships-used.get(p.id,0)-gn(p)

    # Quick scoring
    cands=[]
    for src in mine:
        if sp(src)<3: continue
        for tgt in others:
            n=capture_n(tgt,av,src.x,src.y)
            if n>sp(src): continue
            _,dd,eta=icp(src.x,src.y,tgt,av,n)
            a,ok=safe(src.x,src.y,math.atan2(tgt.y-src.y,tgt.x-src.x),dd)
            if not ok: continue
            tw=max(0,rem-eta); prod=tgt.production
            sc=(prod**2)*10*tw-(dd*0.5)
            if tgt.owner>=0: sc*=1.5
            cands.append((sc,src,tgt,n,a))

    cands.sort(key=lambda x:-x[0])
    atks=0
    for sc,src,tgt,n,a in cands:
        if atks>=4: break
        if tgt.id in done or sp(src)<n: continue
        moves.append([src.id,a,n])
        used[src.id]=used.get(src.id,0)+n; done.add(tgt.id); atks+=1
    return moves

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6: MCTS AGENT
# Budget: 50ms per move (Kaggle safe)
# ─────────────────────────────────────────────────────────────────────────────
class MCTSAgent:
    def __init__(self, budget_ms=45, rollout_depth=25, n_candidates=8):
        self.budget_ms     = budget_ms
        self.rollout_depth = rollout_depth
        self.n_candidates  = n_candidates

    def get_candidate_moves(self, state, player):
        """
        Generate N diverse candidate move-sets using KRONOS heuristic
        with small variations (different targets, different ship counts).
        """
        base = kronos_rollout(state.planets, state.fleets, player,
                              state.av, state.step)
        candidates = [base]

        # Variations: attack different targets, different allocations
        mine   = [p for p in state.planets if p.owner==player]
        others = [p for p in state.planets if p.owner!=player]
        if not mine or not others: return [base]

        for _ in range(self.n_candidates - 1):
            # Random subset of targets
            n_targets = random.randint(1, min(3, len(others)))
            tgts = random.sample(others, n_targets)
            mv = []
            used = {}
            for tgt in tgts:
                src = max([p for p in mine
                           if p.ships-used.get(p.id,0)-4 > 0],
                          key=lambda p:p.ships-used.get(p.id,0),
                          default=None)
                if src is None: continue
                n = capture_n(tgt,state.av,src.x,src.y)
                sp = src.ships-used.get(src.id,0)-4
                if sp < n: continue
                a,dd,_ = icp(src.x,src.y,tgt,state.av,n)
                sa,ok  = safe(src.x,src.y,a,dd)
                if ok:
                    mv.append([src.id,sa,n])
                    used[src.id]=used.get(src.id,0)+n
            candidates.append(mv)

        return candidates

    def rollout(self, state, player):
        """Simulate game to depth using KRONOS heuristic."""
        gs = state.copy()
        for _ in range(self.rollout_depth):
            if gs.is_terminal(): break
            # All players act
            for pid in range(4):
                mvs = kronos_rollout(gs.planets,gs.fleets,pid,gs.av,gs.step)
                gs.apply_moves(mvs, pid)
            gs.step_sim()
        return gs.score(player)

    def search(self, state, player):
        """Run MCTS within time budget. Returns best move list."""
        t0 = time.time()
        candidates = self.get_candidate_moves(state, player)

        root = MCTSNode(moves=[])
        root.visits = 1
        root.untried = list(range(len(candidates)))

        # Create child nodes for each candidate
        for i, mv in enumerate(candidates):
            child = MCTSNode(moves=mv, parent=root)
            root.children.append(child)

        if not root.children:
            return kronos_rollout(state.planets,state.fleets,player,
                                   state.av,state.step)

        iters = 0
        while (time.time()-t0)*1000 < self.budget_ms:
            # SELECTION: pick best UCB1 child
            node = max(root.children, key=lambda c: c.ucb1())

            # SIMULATION: apply moves and rollout
            sim_state = state.copy()
            sim_state.apply_moves(node.moves, player)
            sim_state.step_sim()
            reward = self.rollout(sim_state, player)

            # BACKPROPAGATION
            node.visits += 1
            node.value  += reward
            root.visits += 1

            iters += 1

        best = root.most_visited()
        return best.moves


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7: GYMNASIUM ENVIRONMENT WRAPPER
# ─────────────────────────────────────────────────────────────────────────────
try:
    import gymnasium as gym
    from gymnasium import spaces

    class OrbitWarsEnv(gym.Env):
        """
        Gymnasium wrapper for orbit_wars.
        Observation: feature vector (280 dims)
        Action: Discrete(n_planets × n_planets × 5 ship_fractions)
        Reward: production_gained - ships_lost_ratio
        """
        metadata = {'render_modes': []}

        def __init__(self, player_id=0):
            super().__init__()
            self.player_id = player_id
            self._make_env()

            # Observation: 20 planets × 14 features + 8 global
            obs_dim = 20*14 + 8
            self.observation_space = spaces.Box(
                low=-1.0, high=2.0,
                shape=(obs_dim,), dtype=np.float32
            )

            # Simplified discrete action: (src_idx, tgt_idx, fraction)
            # fraction: 0=25%, 1=50%, 2=75%, 3=100%, 4=skip
            self.action_space = spaces.MultiDiscrete([20, 20, 5])
            self._prev_my_prod = 0

        def _make_env(self):
            from kaggle_environments import make
            self._env = make('orbit_wars', debug=False)

        def reset(self, seed=None, options=None):
            super().reset(seed=seed)
            self._env.reset()
            obs = self._get_obs()
            self._prev_my_prod = 0
            return obs, {}

        def _get_obs(self):
            raw = dict(self._env.state[0].observation)
            planets = [_P(*p) for p in raw.get('planets',[])]
            fleets  = [_F(*f) for f in raw.get('fleets',[])]
            av      = raw.get('angular_velocity', 0.0366)
            stp     = raw.get('step', 0)
            return extract_features(planets,fleets,self.player_id,av,stp)

        def step(self, action):
            raw = dict(self._env.state[0].observation)
            planets = [_P(*p) for p in raw.get('planets',[])]
            fleets  = [_F(*f) for f in raw.get('fleets',[])]
            av      = raw.get('angular_velocity', 0.0366)
            stp     = raw.get('step', 0)

            mine = [p for p in planets if p.owner==self.player_id]
            others= [p for p in planets if p.owner!=self.player_id]

            # Decode action
            src_idx, tgt_idx, frac_idx = int(action[0]), int(action[1]), int(action[2])
            moves = []
            if (frac_idx < 4 and src_idx < len(mine) and tgt_idx < len(others)):
                src = mine[src_idx]
                tgt = others[tgt_idx]
                fracs = [0.25, 0.5, 0.75, 1.0]
                n = max(1, int(src.ships * fracs[frac_idx]))
                a,dd,_ = icp(src.x,src.y,tgt,av,n)
                sa,ok  = safe(src.x,src.y,a,dd)
                if ok: moves.append([src.id,sa,n])

            # Step environment
            self._env.step([moves, None, None, None])

            # Get next obs
            obs = self._get_obs()

            # Reward
            new_raw = dict(self._env.state[0].observation)
            new_planets = [_P(*p) for p in new_raw.get('planets',[])]
            my_prod = sum(p.production for p in new_planets if p.owner==self.player_id)
            reward  = (my_prod - self._prev_my_prod) * 0.5
            self._prev_my_prod = my_prod

            # Terminal
            done = (self._env.state[0].status != 'ACTIVE')
            if done:
                r = self._env.state[0].reward
                reward += (50.0 if r==1 else -20.0)

            return obs, reward, done, False, {}

    print("✅ Gymnasium OrbitWarsEnv defined")

except ImportError:
    print("⚠️  gymnasium not installed — MCTS agent still works without it")
    OrbitWarsEnv = None

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8: FINAL SUBMISSION AGENT
# Uses MCTS when time allows, falls back to KRONOS v6
# ─────────────────────────────────────────────────────────────────────────────
_mcts_agent = MCTSAgent(budget_ms=45, rollout_depth=20, n_candidates=8)

def orbital_strategist(obs):
    if isinstance(obs,dict):
        pl=obs.get('player',0); rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',0)
    else:
        pl=obs.player; rp=obs.planets; rf=obs.fleets
        av=obs.angular_velocity; stp=getattr(obs,'step',0)

    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
        planets=[NP(*p) for p in rp]; fleets=[NF(*f) for f in rf]
    except Exception:
        planets=[_P(*p) for p in rp]; fleets=[_F(*f) for f in rf]

    mine=[p for p in planets if p.owner==pl]
    if not mine: return []

    # Build game state for MCTS
    state = GameState(planets, fleets, av, stp)

    # MCTS search
    try:
        best_moves = _mcts_agent.search(state, pl)
        return best_moves if best_moves else []
    except Exception:
        # Fallback to pure KRONOS
        return kronos_rollout(planets,fleets,pl,av,stp)

agent = orbital_strategist


## ⏱️ Cell 4 — MCTS Benchmark (speed test)


In [ ]:
import time

# Test MCTS speed
env_test=make('orbit_wars',debug=False)
env_test.reset()
raw=dict(env_test.state[0].observation)
test_planets=[_P(*p) for p in raw['planets']]
test_fleets =[_F(*f) for f in raw.get('fleets',[])]
test_av=raw['angular_velocity']

state=GameState(test_planets,test_fleets,test_av,0)
agent_mcts=MCTSAgent(budget_ms=45,rollout_depth=20,n_candidates=8)

t0=time.time()
moves=agent_mcts.search(state,0)
elapsed=(time.time()-t0)*1000

print(f'⏱️  MCTS completed in {elapsed:.1f}ms')
print(f'   Moves decided: {len(moves)}')
print(f'   Moves: {moves[:2]}...' if len(moves)>2 else f'   Moves: {moves}')
print(f'   Budget: 45ms — {"✅ safe" if elapsed<50 else "⚠️ over budget"}')


## 🔬 Cell 5 — v1 Baseline


In [ ]:
def v1_agent(obs):
    import math
    class _P2:
        __slots__=['id','owner','x','y','radius','ships','production']
        def __init__(self,*a):
            for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
    class _F2:
        __slots__=['id','owner','x','y','angle','ships']
        def __init__(self,*a):
            for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as _P2,Fleet as _F2
    except: pass
    def fs(n): return min(6.0,1.0+(max(1,n)-1)*5.0/99.0)
    def dd(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
    def isin(p): return dd(p.x,p.y,50,50)<38
    def pp(p,av,t):
        if not isin(p): return p.x,p.y
        r=dd(p.x,p.y,50,50); a=math.atan2(p.y-50,p.x-50)+av*t
        return 50+r*math.cos(a),50+r*math.sin(a)
    def icp2(sx,sy,tp,av,n):
        tx,ty=tp.x,tp.y
        for _ in range(15):
            d=dd(sx,sy,tx,ty); t=d/fs(n) if fs(n)>0 else 1e9
            nx,ny=pp(tp,av,t)
            if dd(tx,ty,nx,ny)<0.05: break
            tx,ty=nx,ny
        d=dd(sx,sy,tx,ty); return math.atan2(ty-sy,tx-sx),d,d/fs(n)
    def sh(ox,oy,a,md2):
        dx,dy=math.cos(a),math.sin(a); fx,fy=50-ox,50-oy; t=fx*dx+fy*dy
        return 0<t<md2 and abs(fx*dy-fy*dx)<6.5
    def sa2(ox,oy,a,d2):
        if not sh(ox,oy,a,d2): return a,True
        for i in range(1,13):
            dl=math.radians(25)*i/12
            for s in(1,-1):
                if not sh(ox,oy,a+s*dl,d2): return a+s*dl,True
        return a,False
    if isinstance(obs,dict):
        pl=obs.get('player',0);rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',250)
    else:
        pl=obs.player;rp=obs.planets;rf=obs.fleets
        av=obs.angular_velocity;stp=getattr(obs,'step',250)
    P=[_P2(*p) for p in rp]; F=[_F2(*f) for f in rf]
    mine=[p for p in P if p.owner==pl]; tgts=[p for p in P if p.owner!=pl]
    if not mine or not tgts: return []
    rem=500-stp; moves=[]; cmtd=set(); used={}
    def av2(p): return p.ships-used.get(p.id,0)
    for t in sorted([t for t in tgts if t.id not in cmtd
                     and t.ships<=t.production*4+3],key=lambda t:t.ships):
        bst=None; bs=-1e9
        for src in mine:
            if av2(src)<15: continue
            _,ddv,ta=icp2(src.x,src.y,t,av,t.ships+5)
            sc=t.production/(ddv+1)
            if sc>bs: bs=sc;bst=src;bta=ta
        if bst is None: continue
        n=max(int((t.ships+t.production*bta)*1.3)+1,int(t.ships*1.3)+5)
        if av2(bst)<n: continue
        ang,ddv,_=icp2(bst.x,bst.y,t,av,n);sva,ok=sa2(bst.x,bst.y,ang,ddv)
        if ok: moves.append([bst.id,sva,n]);used[bst.id]=used.get(bst.id,0)+n;cmtd.add(t.id)
    cds=[]
    for src in mine:
        a2v=av2(src)
        if a2v<10: continue
        for t in tgts:
            if t.id in cmtd: continue
            _,ddv,ta=icp2(src.x,src.y,t,av,min(a2v,50))
            n=max(int((t.ships+t.production*ta)*1.3)+1,int(t.ships*1.3)+5)
            if n>a2v or n<=t.ships+t.production*ta: continue
            r=(t.production*max(0,rem-ta)-n)/(ta+1)+t.production*2
            if t.ships<=t.production*4+3: r*=1.5
            cds.append((r,src,t,n,ddv))
    cds.sort(key=lambda x:-x[0])
    for r,src,t,n,ddv in cds:
        if t.id in cmtd or av2(src)<n: continue
        ang,dd2,_=icp2(src.x,src.y,t,av,n);sva,ok=sa2(src.x,src.y,ang,dd2)
        if not ok: continue
        moves.append([src.id,sva,n]);used[src.id]=used.get(src.id,0)+n;cmtd.add(t.id)
    return moves
print('✅ v1 ready')


## 🧪 Cell 6 — MCTS vs v1 (1v1)


In [ ]:
e1=make('orbit_wars',debug=False)
e1.run([orbital_strategist,v1_agent])
r1=[s.reward for s in e1.steps[-1]]
print(f'  {"🏆" if r1[0]==1 else "  "} MCTS : {r1[0]:+d}')
print(f'  {"🏆" if r1[1]==1 else "  "} v1   : {r1[1]:+d}')
e1.render(mode='ipython',width=800,height=600)


## 🎮 Cell 7 — 4-Player


In [ ]:
e4=make('orbit_wars',debug=False)
e4.run([orbital_strategist,v1_agent,'random',v1_agent])
r4=[s.reward for s in e4.steps[-1]]
for lb,rw in zip(['🧠 MCTS','v1-A','🎲','v1-B'],r4):
    print(f'  {"🏆" if rw==1 else "  "} {lb:8s} {rw:+d}')
e4.render(mode='ipython',width=800,height=600)


## 📊 Cell 8 — Tournament 20 Games


In [ ]:
import random as _rnd
N=20; wins={'MCTS':0,'v1':0,'rnd':0}
for g in range(N):
    agents=[orbital_strategist,v1_agent,'random',v1_agent]
    _rnd.shuffle(agents); kp=agents.index(orbital_strategist)
    et=make('orbit_wars',debug=False); et.run(agents)
    rws=[s.reward for s in et.steps[-1]]; w=rws.index(max(rws))
    if w==kp: wins['MCTS']+=1; wl='🧠 MCTS'
    elif agents[w]==v1_agent: wins['v1']+=1; wl='v1'
    else: wins['rnd']+=1; wl='🎲'
    print(f'G{g+1:02d}[M@{kp}] {[f"{r:+d}" for r in rws]} → {wl}')
print('─'*50)
for nm,w in wins.items(): print(f'  {nm:5s}: {w}/{N}  {"█"*(w*2)}')
wr=wins['MCTS']/N; elo=int(600+max(0,wr-0.25)*3800)
print(f'\n  Win rate : {wr:.0%}  |  Elo est: ~{elo}')
print(f'  {"🏆 TOP 3 CONTENDER!" if elo>=1400 else "✅ Competitive" if elo>=1000 else "⚠️"}')


## 🤖 Cell 9 — PPO Self-Play Training (optional)

Run this cell to train a neural network policy from scratch.
Leave running overnight for best results (~1M steps).


In [ ]:
try:
    from stable_baselines3 import PPO
    from stable_baselines3.common.env_util import make_vec_env
    import torch

    if OrbitWarsEnv is not None:
        # Create vectorized env for faster training
        print('🤖 Creating training environment...')
        train_env = OrbitWarsEnv(player_id=0)

        # PPO with MLP policy
        model = PPO(
            'MlpPolicy',
            train_env,
            verbose=1,
            learning_rate=3e-4,
            n_steps=512,
            batch_size=64,
            n_epochs=5,
            gamma=0.995,       # high gamma = values long-term production
            gae_lambda=0.95,
            clip_range=0.2,
            policy_kwargs=dict(
                net_arch=[256, 256, 128],   # 3-layer MLP
                activation_fn=torch.nn.ReLU
            )
        )

        print('📐 Policy architecture:')
        print(model.policy)
        print(f'\n   Observation dim : {train_env.observation_space.shape}')
        print(f'   Action space     : {train_env.action_space}')
        print('\n🚀 To train: model.learn(total_timesteps=500_000)')
        print('💾 To save : model.save("kronos_ppo")')
        print('📂 To load : model = PPO.load("kronos_ppo")')
    else:
        print('⚠️  Install gymnasium to use PPO training')

except ImportError as e:
    print(f'⚠️  {e}')
    print('   Run: !pip install stable-baselines3 torch')


## 💾 Cell 10 — Write main.py


In [ ]:
%%writefile main.py
"""
KRONOS MCTS — Monte Carlo Tree Search + KRONOS v6 Rollout Policy
================================================================
Architecture:
  1. OrbitWarsEnv      — Gymnasium wrapper for orbit_wars
  2. FeatureExtractor  — Strategic feature engineering (not just distance)
  3. MCTSNode          — Tree node with UCB1 selection
  4. MCTSAgent         — 50ms budget MCTS with KRONOS rollout
  5. PPOTrainer        — Self-play training setup (Stable Baselines3)

Why MCTS > pure heuristic:
  - Simulates 200-500 future states per move
  - Picks the action that wins in MOST simulated futures
  - KRONOS v6 used as rollout policy (fast approximation)
  - UCB1 balances exploration vs exploitation
"""
import math, time, random, copy, collections
import numpy as np

SX,SY,SR,INNER,MS = 50.0,50.0,5.0,38.0,500

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1: PHYSICS (shared)
# ─────────────────────────────────────────────────────────────────────────────
class _P:
    __slots__=['id','owner','x','y','radius','ships','production']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
    def copy(self):
        p=_P(); [setattr(p,f,getattr(self,f)) for f in self.__slots__]; return p

class _F:
    __slots__=['id','owner','x','y','angle','ships']
    def __init__(self,*a):
        for i,f in enumerate(self.__slots__): setattr(self,f,a[i] if i<len(a) else 0)
    def copy(self):
        f=_F(); [setattr(f,s,getattr(self,s)) for s in self.__slots__]; return f

def spd(n):  return min(6.0,1.0+(max(1,n)-1)*5.0/99.0)
def d2(ax,ay,bx,by): return math.sqrt((ax-bx)**2+(ay-by)**2)
def inn(p):  return d2(p.x,p.y,SX,SY)<INNER
def pred(p,av,t):
    if not inn(p): return p.x,p.y
    r=d2(p.x,p.y,SX,SY); a=math.atan2(p.y-SY,p.x-SX)+av*t
    return SX+r*math.cos(a),SY+r*math.sin(a)
def icp(sx,sy,tp,av,n,it=18):
    tx,ty=tp.x,tp.y
    for _ in range(it):
        dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
        nx,ny=pred(tp,av,t)
        if d2(tx,ty,nx,ny)<0.02: break
        tx,ty=(tx+nx)/2,(ty+ny)/2
    dd=d2(sx,sy,tx,ty); t=dd/spd(n) if spd(n)>0 else 1e9
    return math.atan2(ty-sy,tx-sx),dd,t
def sun_ok(ox,oy,a,md):
    dx,dy=math.cos(a),math.sin(a); fx,fy=SX-ox,SY-oy; tp=fx*dx+fy*dy
    if not(0<tp<md): return True
    return abs(fx*dy-fy*dx)>=SR+1.5
def safe(ox,oy,a,d,sw=42,st=32):
    if sun_ok(ox,oy,a,d): return a,True
    for i in range(1,st+1):
        da=math.radians(sw)*i/st
        for s in(+1,-1):
            alt=a+s*da
            if sun_ok(ox,oy,alt,d): return alt,True
    return a,False
def capture_n(tgt,av,sx,sy,buf=1.07):
    lo,hi=1,max(tgt.ships*2+20,30)
    for _ in range(14):
        mid=(lo+hi)//2
        _,_,eta=icp(sx,sy,tgt,av,mid)
        if mid>int((tgt.ships+tgt.production*eta)*buf): hi=mid
        else: lo=mid+1
    return hi

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2: FEATURE EXTRACTOR
# Strategic features far richer than raw coordinates
# ─────────────────────────────────────────────────────────────────────────────
def extract_features(planets, fleets, player, av, stp):
    """
    Returns numpy array of features for RL observation space.
    Per-planet features (14 features × max_planets):
      - owner_one_hot (4), ships_norm, production_norm, 
        dist_to_sun_norm, dist_to_centroid_norm,
        is_inner, threat_incoming_norm, threat_eta_norm,
        strategic_value, control_radius
    Global features (8):
      - my_prod_share, my_ship_share, step_norm,
        neutral_count_norm, enemy_count_norm,
        my_planet_count_norm, winning_flag, prod_lead
    """
    MAX_P = 20
    mine   = [p for p in planets if p.owner==player]
    enemy  = [p for p in planets if p.owner>=0 and p.owner!=player]
    neutral= [p for p in planets if p.owner<0]

    if mine:
        cx = sum(p.x for p in mine)/len(mine)
        cy = sum(p.y for p in mine)/len(mine)
    else:
        cx, cy = SX, SY

    total_ships = max(1, sum(p.ships for p in planets if p.owner>=0))
    total_prod  = max(1, sum(p.production for p in planets))
    my_ships    = sum(p.ships for p in mine)
    my_prod     = sum(p.production for p in mine)

    # Per-planet features
    planet_feats = np.zeros((MAX_P, 14), dtype=np.float32)
    for i, p in enumerate(planets[:MAX_P]):
        # Owner one-hot (4 players: -1=neutral mapped to 0, else 1-4)
        owner_idx = max(0, min(3, p.owner+1)) if p.owner>=0 else 0
        planet_feats[i, owner_idx] = 1.0
        planet_feats[i, 4]  = p.ships / 200.0
        planet_feats[i, 5]  = p.production / 10.0
        planet_feats[i, 6]  = d2(p.x,p.y,SX,SY) / 70.0
        planet_feats[i, 7]  = d2(p.x,p.y,cx,cy) / 100.0
        planet_feats[i, 8]  = 1.0 if inn(p) else 0.0

        # Threat
        thr = sum(f.ships for f in fleets
                  if f.owner!=player
                  and icp(f.x,f.y,p,av,f.ships)[1] < p.radius+5
                  and icp(f.x,f.y,p,av,f.ships)[2] < 30)
        planet_feats[i, 9]  = min(1.0, thr / 100.0)

        # Strategic value: production × (1/distance_to_enemy_centroid)
        if enemy:
            ecx=sum(e.x for e in enemy)/len(enemy)
            ecy=sum(e.y for e in enemy)/len(enemy)
            planet_feats[i,10] = p.production / max(1,d2(p.x,p.y,ecx,ecy)/10)
        planet_feats[i,11] = (p.owner==player)*1.0
        planet_feats[i,12] = 1.0 if p.owner<0 else 0.0
        planet_feats[i,13] = p.production / 10.0 * (1.0 if p.owner!=player else 0.0)

    # Global features
    global_feats = np.array([
        my_prod / total_prod,                           # production share
        my_ships / total_ships,                         # ship share
        stp / MS,                                       # game progress
        len(neutral) / max(1,len(planets)),             # neutral ratio
        len(enemy) / 4.0,                               # enemy planet ratio
        len(mine) / max(1,len(planets)),                # my planet ratio
        1.0 if my_ships > total_ships/4*1.1 else 0.0,  # winning flag
        (my_prod - (total_prod-my_prod)/3) / 10.0,     # production lead
    ], dtype=np.float32)

    return np.concatenate([planet_feats.flatten(), global_feats])

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3: LIGHTWEIGHT GAME SIMULATOR (for MCTS rollouts)
# ─────────────────────────────────────────────────────────────────────────────
class GameState:
    """
    Minimal game state for fast simulation.
    Simulates one turn: fleet movement, planet capture, production.
    """
    def __init__(self, planets, fleets, av, step, n_players=4):
        self.planets   = [p.copy() for p in planets]
        self.fleets    = [f.copy() for f in fleets]
        self.av        = av
        self.step      = step
        self.n_players = n_players
        self._fid      = max((f.id for f in fleets), default=0) + 1

    def apply_moves(self, moves, player):
        """Apply move list from agent: [[src_id, angle, n_ships], ...]"""
        pid_map = {p.id:p for p in self.planets}
        for move in moves:
            src_id, angle, n = move[0], move[1], move[2]
            src = pid_map.get(src_id)
            if src is None or src.ships < n: continue
            src.ships -= n
            nf = _F()
            nf.id=self._fid; nf.owner=player
            nf.x=src.x; nf.y=src.y
            nf.angle=angle; nf.ships=n
            self.fleets.append(nf)
            self._fid += 1

    def step_sim(self):
        """Advance game by one turn."""
        # Move fleets
        alive = []
        for f in self.fleets:
            s = spd(f.ships)
            f.x += math.cos(f.angle)*s
            f.y += math.sin(f.angle)*s
            if d2(f.x,f.y,SX,SY) < SR: continue
            if not(0<=f.x<=100 and 0<=f.y<=100): continue
            alive.append(f)
        self.fleets = alive

        # Rotate inner planets
        for p in self.planets:
            if inn(p):
                r=d2(p.x,p.y,SX,SY); a=math.atan2(p.y-SY,p.x-SX)+self.av
                p.x=SX+r*math.cos(a); p.y=SY+r*math.sin(a)

        # Fleet arrivals
        arrived=[]
        for f in self.fleets:
            for p in self.planets:
                if d2(f.x,f.y,p.x,p.y) < p.radius + spd(f.ships) + 0.5:
                    if f.owner==p.owner: p.ships+=f.ships
                    else:
                        p.ships-=f.ships
                        if p.ships<0: p.owner=f.owner; p.ships=abs(p.ships)
                    arrived.append(f); break
        for f in arrived:
            if f in self.fleets: self.fleets.remove(f)

        # Production
        for p in self.planets:
            if p.owner>=0: p.ships+=p.production

        self.step+=1

    def score(self, player):
        """Evaluate position for player. Higher = better."""
        my_ships = sum(p.ships for p in self.planets if p.owner==player)
        my_prod  = sum(p.production for p in self.planets if p.owner==player)
        my_fleet = sum(f.ships for f in self.fleets if f.owner==player)
        total    = max(1, sum(p.ships for p in self.planets if p.owner>=0))
        return (my_ships+my_fleet)/total + my_prod*0.3

    def is_terminal(self):
        return self.step >= MS

    def copy(self):
        gs = GameState.__new__(GameState)
        gs.planets   = [p.copy() for p in self.planets]
        gs.fleets    = [f.copy() for f in self.fleets]
        gs.av        = self.av
        gs.step      = self.step
        gs.n_players = self.n_players
        gs._fid      = self._fid
        return gs

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4: MCTS NODE
# UCB1: score = win_rate + C × sqrt(ln(parent_visits) / node_visits)
# ─────────────────────────────────────────────────────────────────────────────
class MCTSNode:
    __slots__ = ['moves','parent','children','visits','value','untried']
    C = 1.414   # exploration constant

    def __init__(self, moves, parent=None):
        self.moves    = moves     # the action that led here
        self.parent   = parent
        self.children = []
        self.visits   = 0
        self.value    = 0.0
        self.untried  = None      # set lazily

    def ucb1(self):
        if self.visits == 0: return float('inf')
        exploit = self.value / self.visits
        explore = self.C * math.sqrt(math.log(self.parent.visits) / self.visits)
        return exploit + explore

    def best_child(self):
        return max(self.children, key=lambda c: c.ucb1())

    def most_visited(self):
        return max(self.children, key=lambda c: c.visits)

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5: KRONOS v6 ROLLOUT POLICY (fast heuristic for simulation)
# ─────────────────────────────────────────────────────────────────────────────
def kronos_rollout(planets, fleets, player, av, stp):
    """Fast KRONOS heuristic for MCTS rollout."""
    mine   = [p for p in planets if p.owner==player]
    others = [p for p in planets if p.owner!=player]
    if not mine or not others: return []

    rem=MS-stp; moves=[]; used={}; done=set()
    def gn(p): return max(3,p.production*2)
    def sp(p): return p.ships-used.get(p.id,0)-gn(p)

    # Quick scoring
    cands=[]
    for src in mine:
        if sp(src)<3: continue
        for tgt in others:
            n=capture_n(tgt,av,src.x,src.y)
            if n>sp(src): continue
            _,dd,eta=icp(src.x,src.y,tgt,av,n)
            a,ok=safe(src.x,src.y,math.atan2(tgt.y-src.y,tgt.x-src.x),dd)
            if not ok: continue
            tw=max(0,rem-eta); prod=tgt.production
            sc=(prod**2)*10*tw-(dd*0.5)
            if tgt.owner>=0: sc*=1.5
            cands.append((sc,src,tgt,n,a))

    cands.sort(key=lambda x:-x[0])
    atks=0
    for sc,src,tgt,n,a in cands:
        if atks>=4: break
        if tgt.id in done or sp(src)<n: continue
        moves.append([src.id,a,n])
        used[src.id]=used.get(src.id,0)+n; done.add(tgt.id); atks+=1
    return moves

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6: MCTS AGENT
# Budget: 50ms per move (Kaggle safe)
# ─────────────────────────────────────────────────────────────────────────────
class MCTSAgent:
    def __init__(self, budget_ms=45, rollout_depth=25, n_candidates=8):
        self.budget_ms     = budget_ms
        self.rollout_depth = rollout_depth
        self.n_candidates  = n_candidates

    def get_candidate_moves(self, state, player):
        """
        Generate N diverse candidate move-sets using KRONOS heuristic
        with small variations (different targets, different ship counts).
        """
        base = kronos_rollout(state.planets, state.fleets, player,
                              state.av, state.step)
        candidates = [base]

        # Variations: attack different targets, different allocations
        mine   = [p for p in state.planets if p.owner==player]
        others = [p for p in state.planets if p.owner!=player]
        if not mine or not others: return [base]

        for _ in range(self.n_candidates - 1):
            # Random subset of targets
            n_targets = random.randint(1, min(3, len(others)))
            tgts = random.sample(others, n_targets)
            mv = []
            used = {}
            for tgt in tgts:
                src = max([p for p in mine
                           if p.ships-used.get(p.id,0)-4 > 0],
                          key=lambda p:p.ships-used.get(p.id,0),
                          default=None)
                if src is None: continue
                n = capture_n(tgt,state.av,src.x,src.y)
                sp = src.ships-used.get(src.id,0)-4
                if sp < n: continue
                a,dd,_ = icp(src.x,src.y,tgt,state.av,n)
                sa,ok  = safe(src.x,src.y,a,dd)
                if ok:
                    mv.append([src.id,sa,n])
                    used[src.id]=used.get(src.id,0)+n
            candidates.append(mv)

        return candidates

    def rollout(self, state, player):
        """Simulate game to depth using KRONOS heuristic."""
        gs = state.copy()
        for _ in range(self.rollout_depth):
            if gs.is_terminal(): break
            # All players act
            for pid in range(4):
                mvs = kronos_rollout(gs.planets,gs.fleets,pid,gs.av,gs.step)
                gs.apply_moves(mvs, pid)
            gs.step_sim()
        return gs.score(player)

    def search(self, state, player):
        """Run MCTS within time budget. Returns best move list."""
        t0 = time.time()
        candidates = self.get_candidate_moves(state, player)

        root = MCTSNode(moves=[])
        root.visits = 1
        root.untried = list(range(len(candidates)))

        # Create child nodes for each candidate
        for i, mv in enumerate(candidates):
            child = MCTSNode(moves=mv, parent=root)
            root.children.append(child)

        if not root.children:
            return kronos_rollout(state.planets,state.fleets,player,
                                   state.av,state.step)

        iters = 0
        while (time.time()-t0)*1000 < self.budget_ms:
            # SELECTION: pick best UCB1 child
            node = max(root.children, key=lambda c: c.ucb1())

            # SIMULATION: apply moves and rollout
            sim_state = state.copy()
            sim_state.apply_moves(node.moves, player)
            sim_state.step_sim()
            reward = self.rollout(sim_state, player)

            # BACKPROPAGATION
            node.visits += 1
            node.value  += reward
            root.visits += 1

            iters += 1

        best = root.most_visited()
        return best.moves


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7: GYMNASIUM ENVIRONMENT WRAPPER
# ─────────────────────────────────────────────────────────────────────────────
try:
    import gymnasium as gym
    from gymnasium import spaces

    class OrbitWarsEnv(gym.Env):
        """
        Gymnasium wrapper for orbit_wars.
        Observation: feature vector (280 dims)
        Action: Discrete(n_planets × n_planets × 5 ship_fractions)
        Reward: production_gained - ships_lost_ratio
        """
        metadata = {'render_modes': []}

        def __init__(self, player_id=0):
            super().__init__()
            self.player_id = player_id
            self._make_env()

            # Observation: 20 planets × 14 features + 8 global
            obs_dim = 20*14 + 8
            self.observation_space = spaces.Box(
                low=-1.0, high=2.0,
                shape=(obs_dim,), dtype=np.float32
            )

            # Simplified discrete action: (src_idx, tgt_idx, fraction)
            # fraction: 0=25%, 1=50%, 2=75%, 3=100%, 4=skip
            self.action_space = spaces.MultiDiscrete([20, 20, 5])
            self._prev_my_prod = 0

        def _make_env(self):
            from kaggle_environments import make
            self._env = make('orbit_wars', debug=False)

        def reset(self, seed=None, options=None):
            super().reset(seed=seed)
            self._env.reset()
            obs = self._get_obs()
            self._prev_my_prod = 0
            return obs, {}

        def _get_obs(self):
            raw = dict(self._env.state[0].observation)
            planets = [_P(*p) for p in raw.get('planets',[])]
            fleets  = [_F(*f) for f in raw.get('fleets',[])]
            av      = raw.get('angular_velocity', 0.0366)
            stp     = raw.get('step', 0)
            return extract_features(planets,fleets,self.player_id,av,stp)

        def step(self, action):
            raw = dict(self._env.state[0].observation)
            planets = [_P(*p) for p in raw.get('planets',[])]
            fleets  = [_F(*f) for f in raw.get('fleets',[])]
            av      = raw.get('angular_velocity', 0.0366)
            stp     = raw.get('step', 0)

            mine = [p for p in planets if p.owner==self.player_id]
            others= [p for p in planets if p.owner!=self.player_id]

            # Decode action
            src_idx, tgt_idx, frac_idx = int(action[0]), int(action[1]), int(action[2])
            moves = []
            if (frac_idx < 4 and src_idx < len(mine) and tgt_idx < len(others)):
                src = mine[src_idx]
                tgt = others[tgt_idx]
                fracs = [0.25, 0.5, 0.75, 1.0]
                n = max(1, int(src.ships * fracs[frac_idx]))
                a,dd,_ = icp(src.x,src.y,tgt,av,n)
                sa,ok  = safe(src.x,src.y,a,dd)
                if ok: moves.append([src.id,sa,n])

            # Step environment
            self._env.step([moves, None, None, None])

            # Get next obs
            obs = self._get_obs()

            # Reward
            new_raw = dict(self._env.state[0].observation)
            new_planets = [_P(*p) for p in new_raw.get('planets',[])]
            my_prod = sum(p.production for p in new_planets if p.owner==self.player_id)
            reward  = (my_prod - self._prev_my_prod) * 0.5
            self._prev_my_prod = my_prod

            # Terminal
            done = (self._env.state[0].status != 'ACTIVE')
            if done:
                r = self._env.state[0].reward
                reward += (50.0 if r==1 else -20.0)

            return obs, reward, done, False, {}

    print("✅ Gymnasium OrbitWarsEnv defined")

except ImportError:
    print("⚠️  gymnasium not installed — MCTS agent still works without it")
    OrbitWarsEnv = None

# ─────────────────────────────────────────────────────────────────────────────
# SECTION 8: FINAL SUBMISSION AGENT
# Uses MCTS when time allows, falls back to KRONOS v6
# ─────────────────────────────────────────────────────────────────────────────
_mcts_agent = MCTSAgent(budget_ms=45, rollout_depth=20, n_candidates=8)

def orbital_strategist(obs):
    if isinstance(obs,dict):
        pl=obs.get('player',0); rp=obs.get('planets',[])
        rf=obs.get('fleets',[]); av=obs.get('angular_velocity',0.0366)
        stp=obs.get('step',0)
    else:
        pl=obs.player; rp=obs.planets; rf=obs.fleets
        av=obs.angular_velocity; stp=getattr(obs,'step',0)

    try:
        from kaggle_environments.envs.orbit_wars.orbit_wars import Planet as NP,Fleet as NF
        planets=[NP(*p) for p in rp]; fleets=[NF(*f) for f in rf]
    except Exception:
        planets=[_P(*p) for p in rp]; fleets=[_F(*f) for f in rf]

    mine=[p for p in planets if p.owner==pl]
    if not mine: return []

    # Build game state for MCTS
    state = GameState(planets, fleets, av, stp)

    # MCTS search
    try:
        best_moves = _mcts_agent.search(state, pl)
        return best_moves if best_moves else []
    except Exception:
        # Fallback to pure KRONOS
        return kronos_rollout(planets,fleets,pl,av,stp)

agent = orbital_strategist


## ✅ Cell 11 — Verify Submission


In [ ]:
import importlib.util
spec=importlib.util.spec_from_file_location('main','main.py')
mod=importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
sub=mod.agent; print(f'✅ {sub.__name__} loaded')
ev=make('orbit_wars',debug=False)
ev.run([sub,v1_agent,'random',v1_agent])
fr=[s.reward for s in ev.steps[-1]]
print(f'Rewards: {fr}')
print('🏆 WINS!' if fr[0]==1 else '✅ Runs correctly')
